# 7. Allocation

Where the values of fields live: **allocators** create arrays and fields, following the [creation functions](https://data-apis.org/array-api/latest/API_specification/creation_functions.html) of the Python Array API standard. Each backend provides one, so fields can be created to suit it, e.g. in its memory layout.

Contents: arrays and array allocators · field allocators and providers · memory layouts and the `column_major` backend.

## Arrays and array allocators

| Concept | Kind | Meaning |
|---|---|---|
| `DType`, `Device` | aliases | A dtype and a device of an array library (simplified: `Any`) |
| `Shape` | alias | `int \| tuple[int, ...]` |
| `NDArray` | protocol | An array of the standard: `shape`, `dtype`, `__array_namespace__()`, e.g. a `numpy.ndarray` |
| `NDArrayAllocator` | protocol | Any object, including a module, with `empty`, `zeros`, `ones`, `full` and their `*_like` variants, taking a shape and returning `NDArray`s, e.g. `numpy` |
| `ArrayNamespace` | protocol | An `NDArrayAllocator` with `__array_api_version__`: an implementation of the whole standard (only these members are listed), e.g. `numpy` |

The numpy type stubs do not declare `__array_api_version__`, so mypy accepts the `numpy` module as an `NDArrayAllocator` but not as an `ArrayNamespace`, although it is one at runtime.

In [1]:
import numpy as np
from concepts import ArrayNamespace, NDArray, NDArrayAllocator, NDArrayFieldAllocator

print(
    "numpy:",
    [
        p.__name__
        for p in (ArrayNamespace, NDArrayAllocator, NDArrayFieldAllocator)
        if isinstance(np, p)
    ],
)
array = np.full((2, 3), 7.0)
print(
    "an NDArray:",
    isinstance(array, NDArray),
    array.shape,
    array.dtype,
    "| namespace:",
    array.__array_namespace__().__name__,
)
# `numpy` also passes the runtime check of `NDArrayFieldAllocator`: the member names are the
# same; only a type checker tells them apart, by the argument and return types.

numpy: ['ArrayNamespace', 'NDArrayAllocator', 'NDArrayFieldAllocator']
an NDArray: True (2, 3) float64 | namespace: numpy


## Field allocators and providers

| Concept | Kind | Meaning |
|---|---|---|
| `NDArrayFieldAllocator` | protocol | The same creation functions, taking a `Domain` instead of a shape and returning `NDArrayField`s: `allocator.zeros({I: range(0, 4)})` |
| `Allocator` | alias | `NDArrayAllocator \| NDArrayFieldAllocator` |
| `AllocatorProvider` | protocol | Any object providing an `Allocator` in `__gt_allocator__`, e.g. a backend |

In `implementation.py`, `NumPyFieldAllocator(order)` creates `NumPyArrayField`s, and every backend is an `AllocatorProvider`. Domains can have index sets, e.g. a field on some cells only, and neighbor dimensions, e.g. a local field.

In [2]:
from concepts import AllocatorProvider, Axis, Location, Neighbor, Relation
from implementation import eager, lazy_backward, numpy_field_allocator


class Cell(Location): ...


class Edge(Location): ...


class K(Axis): ...


class C2E(Relation[Cell, Edge], max_neighbors=3): ...


print(
    "eager: AllocatorProvider",
    isinstance(eager, AllocatorProvider),
    "| allocator",
    eager.__gt_allocator__,
)
print("numpy: AllocatorProvider", isinstance(np, AllocatorProvider))  # no `__gt_allocator__`
allocator = lazy_backward.__gt_allocator__
print(
    "NDArrayFieldAllocator:",
    isinstance(allocator, NDArrayFieldAllocator),
    "| same:",
    allocator is numpy_field_allocator,
)
columns = allocator.zeros({Cell: range(0, 3), K: range(0, 2)})
print("zeros     =", columns)
print("full      =", allocator.full({Cell: (0, 4, 9)}, 1.5))  # on an index set
print("ones_like =", allocator.ones_like(columns, dtype=np.int32))
# a local field
weights = allocator.full({Cell: range(0, 2), Neighbor[C2E]: range(0, 3)}, 1.0 / 3.0)
print("local     =", weights.domain[Neighbor[C2E]], weights.data.shape)

eager: AllocatorProvider True | allocator NumPyFieldAllocator(order='C')
numpy: AllocatorProvider False
NDArrayFieldAllocator: True | same: True
zeros     = Field(Cell[0:3], K[0:2]: [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]])
full      = Field(Cell{0, 4, 9}: [1.5, 1.5, 1.5])
ones_like = Field(Cell[0:3], K[0:2]: [[1, 1], [1, 1], [1, 1]])
local     = range(0, 3) (2, 3)


## Memory layouts and the `column_major` backend

The memory layout of the values is the allocator's choice, not part of the concepts: `numpy_field_allocator` lays them out row-major ("C" order: the last dimension varies fastest), and `column_major_field_allocator` column-major ("F" order: the first one does). The `column_major` backend computes like `eager`, and stores its results, field operators and scans alike, in fields from its allocator (`store`). The values are the same; only their strides change.

In a running local program (see "10. Local programs"), fields are values: `empty`, `empty_like` and writes `f[...] = ...` raise a `RuntimeError`. `store` allocates with `zeros_like` and writes through a private path, so `column_major` works there too.

In [3]:
from concepts import Dims, Field
from implementation import (
    NumPyArrayField,
    column_major,
    column_major_field_allocator,
    field_operator,
    scan_operator,
    store,
)


def layout(field):
    """The memory order of the values of `field`, and their strides in bytes."""
    flags = field.data.flags
    return (
        "C" if flags.c_contiguous else "F" if flags.f_contiguous else "strided"
    ) + f" {field.data.strides}"


@field_operator()
def warming(t: Field[Dims[Cell, K], float], rate: float) -> Field[Dims[Cell, K], float]:
    return t + rate * t(K + 1) - rate * t


t = NumPyArrayField({Cell: range(0, 3), K: range(0, 4)}, 280.0 + np.arange(12.0).reshape(3, 4))
results = {}
for backend in (eager, column_major):
    allocator = backend.__gt_allocator__
    t_in = allocator.empty_like(t)
    t_in[...] = t  # the values of `t`, in the backend's layout
    results[backend] = result = backend(warming)(t_in, 0.5)
    print(f"--- {backend}: {allocator} | input {layout(t_in)} | result {layout(result)}")
print("same values:", np.array_equal(results[eager].data, results[column_major].data))
print(
    "store:",
    layout(store(t, column_major_field_allocator)),
    "| a tuple:",
    [layout(f) for f in store((t, t), column_major_field_allocator)],
)


@scan_operator(axis=K)
def column_total(carry: float, x: float) -> float:
    return carry + x


print("column-major scan:", layout(column_major(column_total)(t)))

[eager] computed Field(Cell[0:3], K[0:3]: [[280.5, 281.5, 282.5], [284.5, 285.5, 286.5], [288.5, 289.5, 290.5]])
--- eager: NumPyFieldAllocator(order='C') | input C (32, 8) | result C (24, 8)
[allocated] computed Field(Cell[0:3], K[0:3]: [[280.5, 281.5, 282.5], [284.5, 285.5, 286.5], [288.5, 289.5, 290.5]])
--- column_major: NumPyFieldAllocator(order='F') | input F (8, 24) | result F (8, 24)
same values: True
store: F (8, 24) | a tuple: ['F (8, 24)', 'F (8, 24)']
[allocated] computed Field(Cell[0:3], K[0:4]: [[280.0, 561.0, 843.0, 1126.0], [284.0, 569.0, 855.0, 1142.0], [288.0, 577.0, 867.0, 1158.0]])
column-major scan: F (8, 24)
